# Tutorial 4: Simulation of Omics-Specific Domain Unregistration in Mouse Thymus

This tutorial uses paired RNA and antibody-derived tag (ADT) profiles from mouse thymus generated by high-resolution Stereo-CITE-seq in the [original study](https://www.biorxiv.org/content/10.1101/2023.04.28.538364v1) and distributed as processed input data through [Zenodo](https://zenodo.org/records/10362607). The processed input comprises 4,697 spatial bins, 23,622 genes and 51 proteins measured on the same tissue section.

In the original [SpatialGlue analysis](https://doi.org/10.1038/s41592-024-02316-4), RNA contributed more to most thymus domains, whereas the middle cortex (cluster 4) was more strongly supported by the protein modality. Here, all ADT profiles in that annotated middle-cortex domain are withheld while RNA remains available. This controlled setting tests whether PRISM can retain spatial-domain structure and impute ADT when the modality carrying the stronger domain signal is absent throughout an entire biological compartment.


In [ ]:
# Environment and imports
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (compute_similarity_prior, plot_prism_imputation_spatial,
                   preprocess_omics, prism_eval_and_save, run_clustering_eval_plot, select_best_device,
                   set_prism_plot_style, set_seed, show_real_missing, simulate_celltype_missing,plot_imputation_metric_boxplot)
set_prism_plot_style()

In [ ]:
# Load data and set up paths
DEVICE = select_best_device()
RANDOM_SEED = 2024
set_seed(RANDOM_SEED)

DATASET_DIR = Path("Datasets") / "mouse thymus"
SOURCE_H5AD = DATASET_DIR / "adata_RNA.h5ad"
TARGET_H5AD = DATASET_DIR / "adata_ADT.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial4_mouse_thymus"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_PATH = RESULTS_DIR / "Mouse_thymus_AOT.npz"
RUN_PREFIX = "Mouse_thymus"

adata_source = sc.read_h5ad(SOURCE_H5AD)
adata_target = sc.read_h5ad(TARGET_H5AD)
adata_source.var_names_make_unique()
adata_target.var_names_make_unique()


Here, `adata_source.obs["reference_domain"]` stores the SpatialGlue-derived reference spatial-domain annotation for each RNA location ([mouse thymus analysis](https://github.com/JinmiaoChenLab/SpatialGlue_notebook/blob/main/Benchmarking_Mouse_Thymus_Fig3a_Fig17a_Fig18a_Fig19a.ipynb)); it defines the middle-cortex ADT holdout and is used for downstream evaluation.


In [ ]:
adata_source

### Simulating domain-specific unregistration

The SpatialGlue-defined middle cortex is used as a domain-level target holdout: ADT is masked at every location assigned to `4-Middle cortex region 2 (DN T, DP T, cTEC)`, whereas RNA remains observed. `missing='0'` denotes an ADT-unregistered location and `missing='1'` an observed RNA-ADT pair.

This tutorial extends the FOV simulations to domain-specific incompleteness, in which the target modality is unavailable throughout an annotated thymus compartment; paired ADT profiles provide ground truth for imputation in this setting.


In [ ]:
# Simulate domain-specific ADT missingness
adata_target, missing_indices, observed_indices = simulate_celltype_missing(
                                                        adata_target, annotation_key="reference_domain",
                                                        missing_celltypes=["4-Middle cortex region 2 (DN T, DP T, cTEC)"],
                                                        missing_key="missing",
                                                        inplace=False)
adata_source.obs["missing"] = 1
show_real_missing(adata_target, spatial_key="spatial", label_key="missing", plot=True, figsize=(4, 4),
                  s=4, title="Protein domain-specific missingness")
print(f"Protein missing cells: {len(missing_indices)}/{adata_target.n_obs}")

In [ ]:
# Preprocessing source (RNA) and target (ADT) modalities
adata_source, _ = preprocess_omics(adata_source, modality="RNA", missing_key="missing", min_cells=10,
                                   hvgs=2500, data_role="source", compute_pca=False, save_raw_eval=False)

adata_target, _ = preprocess_omics(adata_target, modality="ADT", missing_key="missing", data_role="target",
                                   compute_pca=False, save_raw_eval=True)

print("RNA shape after preprocessing:", adata_source.shape)
print("ADT shape after preprocessing:", adata_target.shape)

In [ ]:
# Constructing the RNA similarity prior
distance_matrix, prior_metrics = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH, device=DEVICE,
                                                          covet_k_spatial=6, covet_gene_num=128, spatial_key="spatial",
                                                          missing_key="missing", store_neighbor_index=True)

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=150)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=150)
PRISM.Stats_Spatial_Net(adata_target)

### Training PRISM

PRISM is trained on complete RNA and the available ADT profiles outside the middle cortex, yielding a shared representation for spatial-domain identification and ADT imputation in the held-out domain.


In [ ]:
# Train PRISM on the full graph
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix,
                                                       k_top=5, n_epochs=1000, lr=6e-4,
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX, 
                                                       device=DEVICE, patience=10, min_epochs=50, 
                                                       center_drop_rate=0.1, noise=0.0,
                                                       load_model_path=False, interaction_pca=True)

### Task 1: Spatial-domain identification

In [ ]:
# Cluster the final embedding and evaluate thymus domains
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb_base",
                                                           label_key="reference_domain", cluster_key="PRISM_mclust",
                                                           n_clusters=8, s=15, use_pca=True, align_labels=True, 
                                                           aligned_key="PRISM_mclust_domain", 
                                                           dataset_name="mouse_thymus")

### Task 2: ADT imputation

In [ ]:
# Evaluate ADT imputation in the held-out domain
imputation_results = prism_eval_and_save(truth_adata=adata_target_out, adata=adata_target_out,
                                         save_path=str(RESULTS_DIR), first_name=RUN_PREFIX, 
                                         missing_indices=missing_indices, save_files=False)

In [ ]:
# Visualize representative protein imputation
FEATURE_TO_PLOT = "Mouse-CD4"
fig, axes = plot_prism_imputation_spatial(imputation_results=imputation_results, split1_indices=missing_indices, 
                                                    feature=FEATURE_TO_PLOT, show_missing_only=False,
                                                    highlight_missing=False, figsize=(8, 3))

feature_idx = adata_target.var_names.get_loc(FEATURE_TO_PLOT)
feature_metrics = {metric: round(float(imputation_results["raw"]["per_protein"][metric][feature_idx]), 4)
                   for metric in ("PCC", "SPCC", "MSE")}
print(f"Representative protein {FEATURE_TO_PLOT}: {feature_metrics}")